B will own the feature engineering notebook section. The most important decision here is how to handle missing combine values — a lot of players never did every drill. The right approach is median imputation within position group (not overall), plus an indicator column like 40yd_missing so the model can learn that absence of data is itself a signal. After that, z-score normalize each metric separately, and build a proper PyTorch Dataset class that returns (features, pos_idx, target_wavoe, hit_label). For the hit label, define a "steal" as wAVOE > 5 and pick number ≥ 100 — that's your class-imbalanced rare-event problem that PR-AUC is specifically designed for.

In [2]:
#from google.colab import drive
#drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [3]:
# Read in the file
df = pd.read_excel("/content/drive/MyDrive/SportDataChallenge/nfl_combine_2000_2020 (1).xlsx")
## THIS MIGHT NOT BE RIGHT OTHERWISE it should be fine.


# not figured how to read a google_sheet yet
df[0:4]

,Rnd,Pick,Tm,Player,Pos,Age,To,AP1,PB,St,...,Season,wAVOE,PosGroup,40yd,Vertical,Broad Jump,Cone,Shuttle,Wt,BroadJump
0,1,1,CLE,Courtney Brown,DE,22.0,2005.0,0,0,4,...,2000,NaN,Defense,4.78,NaN,NaN,NaN,NaN,269.0,NaN
1,1,2,WAS,LaVar Arrington,LB,22.0,2006.0,0,3,5,...,2000,NaN,Defense,4.53,NaN,NaN,NaN,NaN,250.0,NaN
2,1,3,WAS,Chris Samuels,T,23.0,2009.0,0,6,9,...,2000,10.714286,Line,5.08,NaN,NaN,NaN,NaN,325.0,NaN
3,1,4,CIN,Peter Warrick,WR,23.0,2005.0,0,0,4,...,2000,-26.904762,Skill,4.58,NaN,NaN,NaN,NaN,194.0,NaN


In [4]:
import pandas as pd
import numpy as np



# cleaning
# Dropping 'Misc' as requested (it's useless)
if 'Misc' in df.columns:
    df.drop(columns=['Misc'], inplace=True)

# Merging duplicate Broad Jump columns
if 'BroadJump' in df.columns:
    df['Broad Jump'] = df['Broad Jump'].fillna(df['BroadJump'])
    df.drop(columns=['BroadJump'], inplace=True)

# Handling International Players
df['College/Univ'] = df['College/Univ'].fillna('International')


# Create indicators BEFORE filling to capture the "absence of data" signal
drills = ['40yd', 'Vertical', 'Broad Jump', 'Cone', 'Shuttle', 'Wt']
for col in drills:
    if col in df.columns:
        df[f'{col}_missing'] = df[col].isna().astype(int)



# Median Imputation for Physical Drills (Potential)
for col in drills:
    if col in df.columns:
        # Fill by Position Median
        df[col] = df[col].fillna(df.groupby('Pos')[col].transform('median'))

# Zero Imputation for Production & Prestige (Results)
# This handles the RB with Interceptions and similar cases automatically
production = [
    'AP1', 'PB', 'St', 'Solo', 'Def_Int', 'Sk', 'Pass_Int',
    'Cmp', 'Pass_Att', 'Pass_Yds', 'Pass_TD', 'Rush_Att',
    'Rush_Yds', 'Rush_TD', 'Rec', 'Rec_Yds', 'Rec_TD', 'G'
]
for col in production:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# for whole positions missing a drill
df = df.fillna(df.median(numeric_only=True))

# Drop rows where we don't have the 'Answer' (Target Variable)
df = df.dropna(subset=['wAV'])



Final Null Count (Should be 0): 0


In [5]:
# Z-score the models by position
# The columns you want to standardize
z_cols = ['40yd', 'Vertical', 'Broad Jump', 'Cone', 'Shuttle', 'Wt']

for col in z_cols:
    if col in df.columns:
        # 'transform' keeps the index the same so it fits right back into your df
        df[f'{col}_z'] = df.groupby('Pos')[col].transform(
            lambda x: (x - x.mean()) / x.std() if x.std() > 0 else 0
        )
df[0:4]

,Rnd,Pick,Tm,Player,Pos,Age,To,AP1,PB,St,...,Broad Jump_missing,Cone_missing,Shuttle_missing,Wt_missing,40yd_z,Vertical_z,Broad Jump_z,Cone_z,Shuttle_z,Wt_z
0,1,1,CLE,Courtney Brown,DE,22.0,2005.0,0,0,4,...,1,1,1,0,-0.182237,-0.104082,-0.085628,-0.076026,-0.030805,-0.016120
1,1,2,WAS,LaVar Arrington,LB,22.0,2006.0,0,3,5,...,1,1,1,0,-1.404355,-0.022129,0.053633,-0.082737,0.059327,0.985751
2,1,3,WAS,Chris Samuels,T,23.0,2009.0,0,6,9,...,1,1,1,0,-0.940888,-0.054991,-0.067266,0.001606,-0.027505,0.793003
3,1,4,CIN,Peter Warrick,WR,23.0,2005.0,0,0,4,...,1,1,1,0,1.174760,0.039709,-0.080949,0.020919,-0.054942,-0.578263


build a proper PyTorch Dataset class that returns (features, pos_idx, target_wavoe, hit_label)

define a "steal" as wAVOE > 5 and pick number ≥ 100

In [6]:
# Create the mapping
pos_list = df['Pos'].unique().tolist()
pos_to_idx = {pos: i for i, pos in enumerate(pos_list)}
df['pos_idx'] = df['Pos'].map(pos_to_idx)

# Define the Hit Label (The "Steal")
# .astype(int) to turn True/False into 1/0
df['hit_label'] = ((df['wAVOE'] > 5) & (df['Pick'] >= 100)).astype(int)

In [7]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class NFLDataset(Dataset):
    def __init__(self, df, feature_cols):
        self.df = df
        # Convert everything to Tensors immediately for speed
        self.features = torch.tensor(df[feature_cols].values, dtype=torch.float32)
        self.pos_idx = torch.tensor(df['pos_idx'].values, dtype=torch.long)
        self.target_wavoe = torch.tensor(df['wAVOE'].values, dtype=torch.float32)
        self.hit_label = torch.tensor(df['hit_label'].values, dtype=torch.float32)

    def __len__(self):
        # how many players are in the set
        return len(self.df)

    def __getitem__(self, idx):
        # It returns a tuple of the 4 items for a specific player
        return (
            self.features[idx],
            self.pos_idx[idx],
            self.target_wavoe[idx],
            self.hit_label[idx]
        )